# 09: Check fundus crops

Finds processed images whose crop missed the fundus: photos saved on a grey or white canvas
(sometimes two eyes side by side), where the old crop kept the whole canvas. A correct crop is
black outside the inscribed circle; this notebook measures, for every processed image, the share
of that outside area that is not black, counts flagged images per site and image size, and shows
a sample of them.

The sheet contains dataset images: keep it out of the repository.

**Settings** (right-hand panel): Accelerator **None**, Internet **On**.
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → **01**, **02**, **03**, **04** (processed image shards)
- *Datasets* → *Your Datasets* → **retinaguard-manifest**

**Run:** *Save Version* → *Save & Run All (Commit)*. About 15 min.

**Output** (`/kaggle/working/crop_check/`): `outside.parquet` (one row per image),
`flagged.jpg`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
FLAG_FRACTION = 0.10  # flagged when more than this share of the outside area is not black
DARK_LUMINANCE = 15

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Measure ------------------------------------------------------------------
import re
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from retinaguard.data.preprocess import circular_mask
from retinaguard.data.processing import ProcessedImages, shard_roots

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/crop_check")
OUT_DIR.mkdir(parents=True, exist_ok=True)
manifests = sorted(INPUT_ROOT.glob("**/manifest.parquet"))
assert manifests, "manifest.parquet not found: attach the retinaguard-manifest dataset"
shard = re.compile(r"^(eyepacs|aptos|idrid|ddr)_\d{3}\.zip$")
shard_dirs = sorted({p.parent for p in INPUT_ROOT.glob("**/*.zip") if shard.match(p.name)})

rows = pd.read_parquet(manifests[0])
rows = rows[rows["processed_path"].notna()]
paths = dict(zip(rows["image_id"], rows["processed_path"], strict=True))
reader = ProcessedImages(paths, shard_roots(*shard_dirs))
outside = None
values = []
start = time.perf_counter()
for i, image_id in enumerate(rows["image_id"]):
    img = reader.read(image_id)
    if outside is None:
        outside = ~circular_mask(img.shape[0], img.shape[0] / 2)
    luminance = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[outside]
    values.append(float((luminance > DARK_LUMINANCE).mean()))
    if i % 10000 == 0:
        print(f"{i} images, {time.perf_counter() - start:.0f} s")
rows = rows.assign(outside_not_black=values)
rows[["image_id", "source", "width", "height", "outside_not_black"]].to_parquet(
    OUT_DIR / "outside.parquet", index=False
)

In [ ]:
# --- 2. Counts -------------------------------------------------------------------
rows["flagged"] = rows["outside_not_black"] > FLAG_FRACTION
print(rows.groupby("source")["flagged"].agg(["size", "sum", "mean"]).round(4).to_string())
print()
flagged = rows[rows["flagged"]]
sizes = flagged.groupby(["source", "width", "height"]).size().sort_values(ascending=False)
print(sizes.head(15).to_string())
print()
print(rows.groupby("source")["outside_not_black"].quantile([0.5, 0.99, 0.999]).unstack().round(3))

In [ ]:
# --- 3. Sample of flagged images -------------------------------------------------
from IPython.display import Image, display

TILE, COLUMNS = 224, 8
groups = [g.sample(min(len(g), 16), random_state=0) for _, g in flagged.groupby("source")]
sample = pd.concat(groups) if groups else flagged
tiles = []
for row in sample.itertuples():
    tile = cv2.resize(reader.read(row.image_id), (TILE, TILE), interpolation=cv2.INTER_AREA)
    caption = f"{row.source} {row.outside_not_black:.2f} {row.width}x{row.height}"
    cv2.rectangle(tile, (0, TILE - 20), (TILE, TILE), (0, 0, 0), thickness=-1)
    cv2.putText(
        tile, caption, (4, TILE - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA
    )
    tiles.append(tile)
if tiles:
    tiles += [np.zeros_like(tiles[0])] * (-len(tiles) % COLUMNS)
    grid = np.vstack([np.hstack(tiles[i : i + COLUMNS]) for i in range(0, len(tiles), COLUMNS)])
    path = OUT_DIR / "flagged.jpg"
    cv2.imwrite(str(path), cv2.cvtColor(grid, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 88])
    display(Image(filename=str(path)))

**After the version finishes:** send the tables of cell 2 and the picture of cell 3.